# 04 train scvi

**Purpose:** Train scVI and calculate the latent representation.

**Original analysis notebook:** `04_scVI_setup_HVG8000_lowmem_corrected.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# scVI setup for GBM core 4-dataset integration — HVG8000 low-memory

This notebook is the corrected scVI step for the current clean pipeline.

**Default input:**

`${GLIOMA_PROJECT_ROOT}/processed/integrated/GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad`

Important: the HVG8000 low-memory object stores raw counts directly in `adata.X`; it intentionally does **not** store a duplicate `adata.layers["counts"]`.  
Therefore scVI is configured with:

```python
layer=None
```

Main outputs are saved under:

`${GLIOMA_PROJECT_ROOT}/processed/integrated/`

and QC tables under:

`${GLIOMA_PROJECT_ROOT}/metadata/`

In [ ]:
# =========================
# 0) Imports and global config
# =========================

import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

import torch
import scvi

warnings.filterwarnings("ignore")

# Reduce CPU thread oversubscription on a 16 GB RAM machine.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
INTEGRATED_DIR = PROCESSED_DIR / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
LOGS_DIR = PROJECT_DIR / "logs"

for d in [INTEGRATED_DIR, METADATA_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Default branch: HVG8000 low-memory.
# If HVG8000 fails later, change N_HVG = 6000 and rerun from top.
# ------------------------------------------------------------------
N_HVG = 8000
RUN_TAG = f"HVG{N_HVG}_lowmem_Xcounts"

INPUT_H5AD_BY_HVG = {
    8000: INTEGRATED_DIR / "GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad",
    6000: INTEGRATED_DIR / "GBM_core_4datasets_HVG6000_pre_scVI_lowmem_Xcounts.h5ad",
    4000: INTEGRATED_DIR / "GBM_core_4datasets_HVG4000_pre_scVI.h5ad",
}

INPUT_H5AD = INPUT_H5AD_BY_HVG[N_HVG]

LATENT_H5AD = INTEGRATED_DIR / f"GBM_core_4datasets_HVG{N_HVG}_scVI_latent_lowmem_Xcounts.h5ad"
UMAP_LEIDEN_H5AD = INTEGRATED_DIR / f"GBM_core_4datasets_HVG{N_HVG}_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad"
MODEL_DIR = INTEGRATED_DIR / f"scVI_model_GBM_core_4datasets_HVG{N_HVG}_lowmem_Xcounts"

# Optional compatibility alias only for the chosen branch.
LATENT_ALIAS_H5AD = INTEGRATED_DIR / f"GBM_core_HVG{N_HVG}_scVI_latent.h5ad"
UMAP_ALIAS_H5AD = INTEGRATED_DIR / f"GBM_core_HVG{N_HVG}_scVI_UMAP_Leiden.h5ad"

SCVI_INPUT_QC_TSV = METADATA_DIR / f"scVI_HVG{N_HVG}_input_qc_lowmem.tsv"
SCVI_TRAINING_CONFIG_TSV = METADATA_DIR / f"scVI_HVG{N_HVG}_training_config_lowmem.tsv"
SCVI_HISTORY_TSV = METADATA_DIR / f"scVI_HVG{N_HVG}_training_history_lowmem.tsv"
SCVI_LATENT_QC_TSV = METADATA_DIR / f"scVI_HVG{N_HVG}_latent_qc_lowmem.tsv"
SCVI_CLUSTER_SIZES_TSV = METADATA_DIR / f"scVI_HVG{N_HVG}_leiden_cluster_sizes_lowmem.tsv"
SCVI_CLUSTER_FRACTIONS_TSV = METADATA_DIR / f"scVI_HVG{N_HVG}_cluster_dataset_condition_fractions_lowmem.tsv"
SCVI_OUTPUT_MANIFEST_TSV = METADATA_DIR / f"scVI_HVG{N_HVG}_output_manifest_lowmem.tsv"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
scvi.settings.seed = SEED

torch.set_float32_matmul_precision("high")

def replace_file(path: Path):
    path = Path(path)
    if path.exists():
        path.unlink()
        print(f"[REPLACED existing file] {path}")

def write_tsv_replace(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    replace_file(path)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path}")

def write_h5ad_replace(adata, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    replace_file(path)
    adata.write_h5ad(path)
    print(f"[SAVED] {path}")

def sparse_or_dense_minmax(X):
    if sp.issparse(X):
        if X.nnz == 0:
            return 0, 0
        return float(X.data.min()), float(X.data.max())
    return float(np.nanmin(X)), float(np.nanmax(X))

def is_integer_like_matrix(X, n_check=500_000, tol=1e-6):
    if sp.issparse(X):
        vals = X.data
    else:
        vals = np.asarray(X).ravel()
    if vals.size == 0:
        return True
    if vals.size > n_check:
        rng = np.random.default_rng(SEED)
        vals = vals[rng.choice(vals.size, size=n_check, replace=False)]
    return bool(np.all(np.abs(vals - np.round(vals)) < tol))

print("scvi-tools version:", scvi.__version__)
print("scanpy version:", sc.__version__)
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("\n[INPUT]", INPUT_H5AD)
print("[LATENT OUTPUT]", LATENT_H5AD)
print("[UMAP/LEIDEN OUTPUT]", UMAP_LEIDEN_H5AD)
print("[MODEL DIR]", MODEL_DIR)

In [ ]:
# =========================
# 1) Read HVG object and validate input
# =========================

if not INPUT_H5AD.exists():
    raise FileNotFoundError(f"Input h5ad not found: {INPUT_H5AD}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

adata = sc.read_h5ad(INPUT_H5AD)

print("Loaded:", adata)
print("n_obs:", adata.n_obs)
print("n_vars:", adata.n_vars)
print("layers:", list(adata.layers.keys()))
print("obs columns:", list(adata.obs.columns)[:80])

required_obs_cols = ["scvi_batch", "core_dataset", "condition", "sample_id", "patient_id"]
missing_required = [c for c in required_obs_cols if c not in adata.obs.columns]
if missing_required:
    raise KeyError(f"Missing required obs columns for scVI: {missing_required}")

if adata.n_vars != N_HVG:
    print(f"[WARNING] Expected {N_HVG} genes, but loaded object has {adata.n_vars} genes.")

x_min, x_max = sparse_or_dense_minmax(adata.X)
x_integer_like = is_integer_like_matrix(adata.X)

print("X min:", x_min)
print("X max:", x_max)
print("X integer-like:", x_integer_like)

if x_min < 0:
    raise ValueError("adata.X contains negative values. scVI requires raw non-negative counts.")

if not x_integer_like:
    raise ValueError(
        "adata.X does not look integer-like. For this low-memory HVG object, "
        "adata.X must contain raw counts. Do not use log-normalized data for scVI."
    )

# Keep metadata as strings/categoricals to avoid hidden issues.
for col in ["scvi_batch", "core_dataset", "condition", "sample_id", "patient_id", "dataset_id", "geo_id", "study_id"]:
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype(str).astype("category")

continuous_covariates = [c for c in ["pct_counts_mt", "pct_counts_ribo"] if c in adata.obs.columns]
categorical_covariates = [c for c in ["core_dataset", "condition"] if c in adata.obs.columns]

input_qc = pd.DataFrame([{
    "input_h5ad": str(INPUT_H5AD),
    "run_tag": RUN_TAG,
    "n_obs": adata.n_obs,
    "n_vars": adata.n_vars,
    "has_counts_layer": "counts" in adata.layers,
    "counts_source_for_scvi": "adata.X",
    "x_min": x_min,
    "x_max": x_max,
    "x_integer_like": x_integer_like,
    "batch_key": "scvi_batch",
    "categorical_covariate_keys": ",".join(categorical_covariates),
    "continuous_covariate_keys": ",".join(continuous_covariates),
    "n_scvi_batches": adata.obs["scvi_batch"].nunique(),
    "n_core_datasets": adata.obs["core_dataset"].nunique(),
    "n_conditions": adata.obs["condition"].nunique(),
    "n_samples": adata.obs["sample_id"].nunique(),
    "n_patients": adata.obs["patient_id"].nunique(),
}])
write_tsv_replace(input_qc, SCVI_INPUT_QC_TSV)

display(input_qc)

print("\nCells by core_dataset:")
display(adata.obs["core_dataset"].value_counts().rename_axis("core_dataset").reset_index(name="n_cells"))

print("\nCells by condition:")
display(adata.obs["condition"].value_counts().rename_axis("condition").reset_index(name="n_cells"))

In [ ]:
# =========================
# 2) Configure scVI
# =========================

# Low-memory HVG objects store raw counts in adata.X, not in layers["counts"].
# Therefore layer=None is intentional and correct here.
BATCH_KEY = "scvi_batch"
CATEGORICAL_COVARIATE_KEYS = categorical_covariates
CONTINUOUS_COVARIATE_KEYS = continuous_covariates

scvi.model.SCVI.setup_anndata(
    adata,
    layer=None,
    batch_key=BATCH_KEY,
    categorical_covariate_keys=CATEGORICAL_COVARIATE_KEYS if len(CATEGORICAL_COVARIATE_KEYS) else None,
    continuous_covariate_keys=CONTINUOUS_COVARIATE_KEYS if len(CONTINUOUS_COVARIATE_KEYS) else None,
)

# Conservative model/training settings for RTX 4050 6GB + 16GB RAM.
# If CUDA OOM happens, reduce BATCH_SIZE to 384 or 256.
N_LATENT = 30
N_HIDDEN = 128
N_LAYERS = 2
DROPOUT_RATE = 0.1
GENE_LIKELIHOOD = "nb"

MAX_EPOCHS = 800
EARLY_STOPPING_PATIENCE = 50
EARLY_STOPPING_MIN_DELTA = 1e-3
CHECK_VAL_EVERY_N_EPOCH = 5
BATCH_SIZE = 512
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES = 1

model = scvi.model.SCVI(
    adata,
    n_hidden=N_HIDDEN,
    n_latent=N_LATENT,
    n_layers=N_LAYERS,
    dropout_rate=DROPOUT_RATE,
    gene_likelihood=GENE_LIKELIHOOD,
)

training_config = pd.DataFrame([{
    "run_tag": RUN_TAG,
    "n_hvg": N_HVG,
    "input_h5ad": str(INPUT_H5AD),
    "layer": "None__adata_X_raw_counts",
    "batch_key": BATCH_KEY,
    "categorical_covariate_keys": ",".join(CATEGORICAL_COVARIATE_KEYS),
    "continuous_covariate_keys": ",".join(CONTINUOUS_COVARIATE_KEYS),
    "n_hidden": N_HIDDEN,
    "n_latent": N_LATENT,
    "n_layers": N_LAYERS,
    "dropout_rate": DROPOUT_RATE,
    "gene_likelihood": GENE_LIKELIHOOD,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
    "batch_size": BATCH_SIZE,
    "accelerator": ACCELERATOR,
    "devices": DEVICES,
    "seed": SEED,
}])
write_tsv_replace(training_config, SCVI_TRAINING_CONFIG_TSV)

print(model)
model.view_anndata_setup()
display(training_config)

In [ ]:
# =========================
# TRAIN scVI MODEL
# max_epochs=800 + early stopping
# =========================

print("Training scVI model")
print("max_epochs:", MAX_EPOCHS)
print("early_stopping:", True)
print("early_stopping_monitor: elbo_validation")
print("early_stopping_patience:", EARLY_STOPPING_PATIENCE)
print("check_val_every_n_epoch:", CHECK_VAL_EVERY_N_EPOCH)

try:
    model.train(
        max_epochs=MAX_EPOCHS,
        train_size=0.9,
        validation_size=0.1,
        accelerator=ACCELERATOR,
        devices=DEVICES,
        early_stopping=True,
        early_stopping_monitor="elbo_validation",
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
        check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
        enable_progress_bar=True,
    )

except TypeError as e:
    print("[WARNING] First training call failed because of scvi-tools version differences:")
    print(repr(e))
    print("Retrying with a more compatible training call...")

    model.train(
        max_epochs=MAX_EPOCHS,
        train_size=0.9,
        validation_size=0.1,
        accelerator=ACCELERATOR,
        devices=DEVICES,
        early_stopping=True,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
        enable_progress_bar=True,
    )


# -------------------------
# Export training history
# -------------------------

try:
    hist = model.history

    if hist is None:
        print("[WARNING] model.history is None")
    else:
        hist_df = pd.DataFrame(hist)

        if len(hist_df) > 0:
            hist_df["epoch"] = np.arange(len(hist_df))
            write_tsv_replace(hist_df, SCVI_HISTORY_TSV)
            display(hist_df.tail(20))
            print(f"[SAVED] {SCVI_HISTORY_TSV}")
        else:
            print("[WARNING] Training history is empty.")

except Exception as e:
    print("[WARNING] Could not export training history:", repr(e))

In [ ]:
# =========================
# 4) Extract latent representation and save model/latent h5ad
# =========================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

latent = model.get_latent_representation()
latent = np.asarray(latent, dtype=np.float32)

adata.obsm["X_scVI"] = latent
adata.uns["scVI_run_info"] = {
    "run_tag": RUN_TAG,
    "n_hvg": int(N_HVG),
    "input_h5ad": str(INPUT_H5AD),
    "counts_source": "adata.X raw counts; layer=None in setup_anndata",
    "model_dir": str(MODEL_DIR),
    "batch_key": BATCH_KEY,
    "categorical_covariate_keys": CATEGORICAL_COVARIATE_KEYS,
    "continuous_covariate_keys": CONTINUOUS_COVARIATE_KEYS,
    "n_latent": int(N_LATENT),
    "n_hidden": int(N_HIDDEN),
    "n_layers": int(N_LAYERS),
    "dropout_rate": float(DROPOUT_RATE),
    "gene_likelihood": GENE_LIKELIHOOD,
    "seed": int(SEED),
}

latent_qc = pd.DataFrame([{
    "run_tag": RUN_TAG,
    "n_obs": adata.n_obs,
    "n_vars": adata.n_vars,
    "X_scVI_shape": str(adata.obsm["X_scVI"].shape),
    "X_scVI_min": float(np.nanmin(adata.obsm["X_scVI"])),
    "X_scVI_max": float(np.nanmax(adata.obsm["X_scVI"])),
    "X_scVI_mean": float(np.nanmean(adata.obsm["X_scVI"])),
    "X_scVI_sd": float(np.nanstd(adata.obsm["X_scVI"])),
    "n_nan": int(np.isnan(adata.obsm["X_scVI"]).sum()),
}])
write_tsv_replace(latent_qc, SCVI_LATENT_QC_TSV)
display(latent_qc)

# Replace model directory if needed
if MODEL_DIR.exists():
    import shutil
    shutil.rmtree(MODEL_DIR)
    print(f"[REPLACED existing model dir] {MODEL_DIR}")

model.save(MODEL_DIR, overwrite=True)
print("[SAVED MODEL]", MODEL_DIR)

# Save latent h5ad
write_h5ad_replace(adata, LATENT_H5AD)

# Compatibility alias
write_h5ad_replace(adata, LATENT_ALIAS_H5AD)

print("Saved latent object:")
print(adata)

In [ ]:
# =========================
# 5) Build neighbors, UMAP, and Leiden clusters on X_scVI
# =========================

# If you restarted after training, uncomment the next line:
# adata = sc.read_h5ad(LATENT_H5AD)

if "X_scVI" not in adata.obsm:
    raise KeyError("X_scVI not found in adata.obsm. Run the latent extraction cell first.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

N_NEIGHBORS = 15
LEIDEN_RESOLUTIONS = [0.1, 0.2, 0.3, 0.4, 0.5]

print("Computing neighbors on X_scVI...")
sc.pp.neighbors(
    adata,
    use_rep="X_scVI",
    n_neighbors=N_NEIGHBORS,
    random_state=SEED,
    key_added="neighbors_scVI",
)

print("Computing UMAP...")
sc.tl.umap(
    adata,
    neighbors_key="neighbors_scVI",
    random_state=SEED,
)

cluster_rows = []

for res in LEIDEN_RESOLUTIONS:
    key = f"leiden_scvi_{res}"
    print(f"Running Leiden {key}...")
    try:
        sc.tl.leiden(
            adata,
            resolution=res,
            key_added=key,
            flavor="igraph",
            n_iterations=2,
            directed=False,
            random_state=SEED,
            neighbors_key="neighbors_scVI",
        )
    except TypeError:
        # Older Scanpy versions may not support neighbors_key in tl.leiden.
        sc.tl.leiden(
            adata,
            resolution=res,
            key_added=key,
            flavor="igraph",
            n_iterations=2,
            directed=False,
            random_state=SEED,
        )

    n_clusters = adata.obs[key].nunique()
    print(key, "n_clusters:", n_clusters)

    sizes = (
        adata.obs[key]
        .astype(str)
        .value_counts()
        .rename_axis("cluster")
        .reset_index(name="n_cells")
    )
    sizes["resolution"] = key
    sizes["run_tag"] = RUN_TAG
    cluster_rows.append(sizes)

cluster_sizes = pd.concat(cluster_rows, ignore_index=True)
cluster_sizes = cluster_sizes[["run_tag", "resolution", "cluster", "n_cells"]]
write_tsv_replace(cluster_sizes, SCVI_CLUSTER_SIZES_TSV)
display(cluster_sizes.head())

In [ ]:
# =========================
# 6) Cluster composition tables by dataset and condition
# =========================

fraction_rows = []

for res in [f"leiden_scvi_{r}" for r in [0.1, 0.2, 0.3, 0.4, 0.5]]:
    if res not in adata.obs.columns:
        continue

    for meta_col in ["core_dataset", "condition", "sample_id", "patient_id"]:
        if meta_col not in adata.obs.columns:
            continue

        tab = pd.crosstab(
            adata.obs[res].astype(str),
            adata.obs[meta_col].astype(str),
            normalize="index",
        )

        tab_long = (
            tab.reset_index()
            .melt(id_vars=res, var_name=meta_col, value_name="fraction")
            .rename(columns={res: "cluster"})
        )
        tab_long["resolution"] = res
        tab_long["metadata_column"] = meta_col
        tab_long["run_tag"] = RUN_TAG
        fraction_rows.append(tab_long)

fractions = pd.concat(fraction_rows, ignore_index=True)
fractions = fractions[["run_tag", "resolution", "metadata_column", "cluster", "fraction"] + 
                      [c for c in fractions.columns if c not in ["run_tag", "resolution", "metadata_column", "cluster", "fraction"]]]

write_tsv_replace(fractions, SCVI_CLUSTER_FRACTIONS_TSV)
display(fractions.head(20))

In [ ]:
# =========================
# 7) Save final UMAP/Leiden h5ad and manifest
# =========================

adata.uns["scVI_umap_leiden_run_info"] = {
    "neighbors_key": "neighbors_scVI",
    "n_neighbors": int(N_NEIGHBORS),
    "leiden_resolutions": LEIDEN_RESOLUTIONS,
    "umap_random_state": int(SEED),
}

write_h5ad_replace(adata, UMAP_LEIDEN_H5AD)

# Compatibility alias
write_h5ad_replace(adata, UMAP_ALIAS_H5AD)

manifest = pd.DataFrame([
    {"name": "input_h5ad", "path": str(INPUT_H5AD), "exists": INPUT_H5AD.exists()},
    {"name": "latent_h5ad", "path": str(LATENT_H5AD), "exists": LATENT_H5AD.exists()},
    {"name": "latent_alias_h5ad", "path": str(LATENT_ALIAS_H5AD), "exists": LATENT_ALIAS_H5AD.exists()},
    {"name": "umap_leiden_h5ad", "path": str(UMAP_LEIDEN_H5AD), "exists": UMAP_LEIDEN_H5AD.exists()},
    {"name": "umap_leiden_alias_h5ad", "path": str(UMAP_ALIAS_H5AD), "exists": UMAP_ALIAS_H5AD.exists()},
    {"name": "model_dir", "path": str(MODEL_DIR), "exists": MODEL_DIR.exists()},
    {"name": "input_qc_tsv", "path": str(SCVI_INPUT_QC_TSV), "exists": SCVI_INPUT_QC_TSV.exists()},
    {"name": "training_config_tsv", "path": str(SCVI_TRAINING_CONFIG_TSV), "exists": SCVI_TRAINING_CONFIG_TSV.exists()},
    {"name": "training_history_tsv", "path": str(SCVI_HISTORY_TSV), "exists": SCVI_HISTORY_TSV.exists()},
    {"name": "latent_qc_tsv", "path": str(SCVI_LATENT_QC_TSV), "exists": SCVI_LATENT_QC_TSV.exists()},
    {"name": "cluster_sizes_tsv", "path": str(SCVI_CLUSTER_SIZES_TSV), "exists": SCVI_CLUSTER_SIZES_TSV.exists()},
    {"name": "cluster_fractions_tsv", "path": str(SCVI_CLUSTER_FRACTIONS_TSV), "exists": SCVI_CLUSTER_FRACTIONS_TSV.exists()},
])
write_tsv_replace(manifest, SCVI_OUTPUT_MANIFEST_TSV)
display(manifest)

print("Final object:")
print(adata)
print("\nUMAP keys:", list(adata.obsm.keys()))
print("Leiden columns:", [c for c in adata.obs.columns if c.startswith("leiden_scvi_")])

In [ ]:
# =========================
# 8) Optional lightweight UMAP plots
# =========================

PLOT_FIGURES = False

if PLOT_FIGURES:
    sc.settings.figdir = str(FIGURES_DIR)

    plot_cols = [
        "core_dataset",
        "condition",
        "leiden_scvi_0.2",
        "leiden_scvi_0.3",
        "leiden_scvi_0.4",
    ]
    plot_cols = [c for c in plot_cols if c in adata.obs.columns]

    sc.pl.umap(
        adata,
        color=plot_cols,
        ncols=2,
        wspace=0.4,
        frameon=False,
        show=False,
        save=f"_{RUN_TAG}_core_dataset_condition_leiden.png",
    )

    print("[SAVED UMAP FIGURES]", FIGURES_DIR)
else:
    print("PLOT_FIGURES=False; skipping plots to avoid heavy rendering in VS Code.")